# Experiment 3: analysis notebook

10060 Physics (Polytechnical Foundation), lab on 25 September 2026.

This notebook reads the CSV files you export from the iOLab app and computes the numbers your report needs: the points A, B and C, the moment the hand force reaches zero, $a_1$ and $a_2$ with uncertainties, and standard scores. It saves report figures and tables in the `output` folder. It writes no report text; the words are yours.

**How to use it**

1. Next to this notebook, make the folders `data/part1/run1`, `data/part1/run2`, and so on, and `data/part2/run1`, `data/part2/run2`, and so on.
2. After every recording, export each chart you need: click the toolbar icon at the bottom left of the chart, then Export to CSV. The iOLab app saves the file in `Documents/iOLab-WorkFiles/export`. Move the files of that recording into the folder of that run. Part 1 needs the accelerometer file and the force file. Part 2 needs the wheel file, plus the accelerometer file if you recorded it.
3. Fill in the settings cell: the mass of your iOLab and the ramp height and length.
4. Run everything (menu Run, then Run All Cells).
5. Look at the plot of every run. If a point or a window looks wrong, change the setting and run again.

If the CSV export fails, read the numbers in the iOLab app, type them into `MANUAL_PART1` and `MANUAL_PART2` in the settings, and run the notebook anyway.

Packages: numpy, pandas, matplotlib. Install with `pip install numpy pandas matplotlib` if they are missing.


In [1]:
import re
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("numpy", np.__version__, "| pandas", pd.__version__)

numpy 2.1.3 | pandas 2.3.3


## Settings

The only cell you normally edit. Every number has a unit in the comment.

In [2]:
# ============================== SETTINGS: edit this cell only ==============================

DATA_DIR = Path("data")              # folder next to this notebook
PART1_DIR = DATA_DIR / "part1"       # one subfolder per run: run1, run2, ... (accelerometer CSV + force CSV)
PART2_DIR = DATA_DIR / "part2"       # one subfolder per run: run1, run2, ... (wheel CSV, optional accelerometer CSV)
OUT_DIR = Path("output")             # figures and tables are written here

g = 9.82                             # m/s^2, value used in the course
m, dm = 0.205, 0.001                 # kg, mass of YOUR iOLab and its uncertainty (weigh it; 0.205 is a typical value)

# Part 2 incline angle. Fill in ONE method (or both). Leave None if not measured.
H, dH = None, None                   # m, height difference between two marks on the ramp, e.g. 0.085, 0.001
L, dL = None, None                   # m, distance between the same two marks measured ALONG the ramp, e.g. 1.000, 0.002

# Analysis choices. The defaults are fine for most runs.
BASELINE_S = 0.4                     # s of quiet data at the start of every recording (wait 1 s before pushing)
SMOOTH_MS = 5                        # ms, light moving average used only to LOCATE points; values use raw data
BAND_K = 2                           # noise band half width, in standard deviations
MARGIN_PUSH_S = 0.15                 # s skipped after the velocity maximum (hand may still touch)
MARGIN_TURN_S = 0.15                 # s skipped on both sides of the turning point
MARGIN_END_S = 0.10                  # s skipped before the iOLab is caught
MANUAL_WINDOWS = {}                  # override Part 2 windows, e.g. {"run3": {"up": (1.20, 1.55), "down": (1.80, 2.60)}}
ACC_AXIS = "auto"                    # Part 1 axis: "auto", "x", "y" or "z" (the push should be along y)
WHEEL_SINGLE_KIND = "auto"           # if a wheel CSV has ONE data column: "auto", "position", "velocity", "acceleration"
REPORT_RUN_PART1 = None              # e.g. "run2" to choose the run used for the report figure (None = first run)
REPORT_RUN_PART2 = None

# Manual mode: use these ONLY if you read the numbers in the iOLab app instead of exporting CSV files.
# Part 1, one dict per run. Every (mean, sd, width) comes from one selection in the app:
#   mean = mu, sd = sigma, width = length of the selected time interval in s.
# MANUAL_PART1 = [dict(run=1, f_acc=800,
#                      a_at_F0=(-1.62, 0.21, 0.010),    # acceleration selection centred on the moment F reaches 0
#                      plateau=(-1.75, 0.20, 0.300),    # acceleration selection on the flat friction part
#                      peak=10.8,                       # acceleration at point A
#                      tA=1.120, tB=1.181, tC=1.201, tF0=1.200)]  # times read with the cursor, s
MANUAL_PART1 = []
# Part 2, one dict per run: selections on the WHEEL ACCELERATION graph.
# MANUAL_PART2 = [dict(run=1, f=100, up=(-0.905, 0.110, 0.40), down=(-0.779, 0.095, 0.80))]
MANUAL_PART2 = []

## Helpers

Reading the iOLab CSV files (the `time` column and the `cal[0]`, `cal[1]`, `cal[2]` columns), rounding with the course rule (uncertainty with one significant figure, two if it starts with 1), and the standard score
$$z=\frac{|x_1-x_2|}{\sqrt{\delta x_1^2+\delta x_2^2}}$$
with the course limits: $z\le 1$ very likely the same, $1<z\le 2$ likely the same, $2<z\le 3$ grey zone, $z>3$ unlikely the same.

In [3]:
# ============================== HELPERS: no need to edit ==============================
OUT_DIR.mkdir(parents=True, exist_ok=True)
BLUE, ORANGE, AQUA, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#8a8984"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e4e3df", "grid.linewidth": 0.5, "font.size": 9})


def fmt(x, dx, unit=""):
    """Round the uncertainty to 1 significant figure (2 if it starts with 1) and the value to match."""
    if x is None or not np.isfinite(x):
        return "n/a"
    if dx is None or not np.isfinite(dx) or dx <= 0:
        return f"{x:.4g} {unit}".strip()
    e = int(math.floor(math.log10(dx)))
    nsig = 2 if dx / 10 ** e < 2 else 1
    dec = nsig - 1 - e
    if dec > 0:
        return f"{x:.{dec}f} ± {dx:.{dec}f} {unit}".strip()
    r = 10 ** (-dec)
    return f"{round(x / r) * r:.0f} ± {round(dx / r) * r:.0f} {unit}".strip()


def zscore(x1, dx1, x2=0.0, dx2=0.0):
    """Standard score |x1 - x2| / sqrt(dx1^2 + dx2^2), as in the course notebook on uncertainties."""
    den = math.sqrt(dx1 ** 2 + dx2 ** 2)
    return abs(x1 - x2) / den if den > 0 else float("inf")


def verdict(z):
    if z <= 1:
        return "very likely the same (z <= 1)"
    if z <= 2:
        return "likely the same (1 < z <= 2)"
    if z <= 3:
        return "grey zone (2 < z <= 3)"
    return "unlikely the same (z > 3)"


def mean_sd_sdom(values):
    v = np.asarray([x for x in values if x is not None and np.isfinite(x)], dtype=float)
    n = v.size
    if n == 0:
        return np.nan, np.nan, np.nan, 0
    sd = v.std(ddof=1) if n > 1 else np.nan
    return v.mean(), sd, (sd / math.sqrt(n) if n > 1 else np.nan), n


def mean_with_floor(values, single_uncs):
    """Mean over runs. Uncertainty = SDOM, but never smaller than the typical single run uncertainty / sqrt(n)."""
    mu, sd, sdom, n = mean_sd_sdom(values)
    floor = math.sqrt(np.mean(np.square(single_uncs))) / math.sqrt(n) if n else np.nan
    unc = floor if (n < 2 or not np.isfinite(sdom)) else max(sdom, floor)
    return mu, unc, n


def moving_average(y, n):
    y = np.asarray(y, dtype=float)
    if n <= 1:
        return y.copy()
    k = np.ones(n) / n
    ypad = np.pad(y, (n // 2, n - 1 - n // 2), mode="edge")
    return np.convolve(ypad, k, mode="valid")


def crossing_time(t, y, level, i_start, direction="down"):
    """First time after index i_start where y crosses `level` (linear interpolation). None if never."""
    y = np.asarray(y)
    for i in range(max(i_start, 1), len(y)):
        if direction == "down" and y[i - 1] > level >= y[i]:
            return t[i - 1] + (level - y[i - 1]) * (t[i] - t[i - 1]) / (y[i] - y[i - 1]), i
        if direction == "up" and y[i - 1] < level <= y[i]:
            return t[i - 1] + (level - y[i - 1]) * (t[i] - t[i - 1]) / (y[i] - y[i - 1]), i
    return None, None


def read_iolab_csv(path):
    """Read one iOLab CSV export. Returns (t, [channels], column_names).
    Handles the iOLab layout (a 'time' column and 'cal[0]', 'cal[1]', 'cal[2]' columns),
    comma or semicolon separators, and decimal point or decimal comma."""
    path = Path(path)
    with open(path, "r", encoding="utf-8-sig", errors="replace") as fh:
        lines = [fh.readline() for _ in range(40)]
    hdr = 0
    for i, ln in enumerate(lines):
        if re.search(r"(?i)\btime\b|cal\s*\[", ln):
            hdr = i
            break
    first = lines[hdr]
    if first.count(";") > first.count(","):
        sep = ";"
    elif first.count("\t") > first.count(","):
        sep = "\t"
    else:
        sep = ","
    df = pd.read_csv(path, sep=sep, skiprows=hdr, dtype=str, encoding="utf-8-sig", engine="python")
    df.columns = [str(c).strip() for c in df.columns]

    def num(col):
        s = df[col].astype(str).str.strip()
        if sep != ",":
            s = s.str.replace(",", ".", regex=False)
        return pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)

    cols = list(df.columns)
    tcol = next((c for c in cols if c.lower() == "time"), None) \
        or next((c for c in cols if re.match(r"(?i)^time\b", c) and "stamp" not in c.lower()), None) \
        or next((c for c in cols if re.match(r"(?i)^t(\s*\(s\))?$", c)), None) \
        or next((c for c in cols if "time" in c.lower()), None)
    if tcol is None:
        raise ValueError(f"{path.name}: no time column found. Columns: {cols}")
    cal = [c for c in cols if re.match(r"(?i)^cal\s*\[\s*\d+\s*\]", c)]
    cal.sort(key=lambda c: int(re.findall(r"\d+", c)[0]))
    if not cal:
        skip = re.compile(r"(?i)(^index$|^idx$|^unnamed|raw|frame|stamp|^sample|^#)")
        cal = [c for c in cols if c != tcol and not skip.search(c) and np.isfinite(num(c)).mean() > 0.9]
    t = num(tcol)
    chans = [num(c) for c in cal]
    ok = np.isfinite(t)
    for c in chans:
        ok &= np.isfinite(c)
    t = t[ok]
    chans = [c[ok] for c in chans]
    order = np.argsort(t, kind="stable")
    t = t[order]
    chans = [c[order] for c in chans]
    if t.size > 2 and np.median(np.diff(t)) > 0.5:      # looks like milliseconds
        t = t / 1000.0
    if t.size < 10 or not chans:
        raise ValueError(f"{path.name}: could not read data columns. Columns: {cols}")
    return t, chans, cal


def show(df):
    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


def rate(t):
    return 1.0 / np.median(np.diff(t))


def looks_like_accelerometer(chans):
    return len(chans) >= 3 and max(abs(np.nanmedian(c)) for c in chans) > 6.0


def csv_files(folder):
    return sorted([p for p in Path(folder).glob("*.csv") if not p.name.startswith(".")])


def run_folders(part_dir):
    part_dir = Path(part_dir)
    if not part_dir.exists():
        return []
    runs = [p for p in part_dir.iterdir() if p.is_dir() and csv_files(p)]
    key = lambda p: (int(re.findall(r"\d+", p.name)[0]) if re.findall(r"\d+", p.name) else 10 ** 6, p.name)
    return sorted(runs, key=key)


print("Helpers loaded. Output folder:", OUT_DIR.resolve())
print("Part 1 runs found:", [p.name for p in run_folders(PART1_DIR)])
print("Part 2 runs found:", [p.name for p in run_folders(PART2_DIR)])

Helpers loaded. Output folder: /Users/anas/Desktop/DTU/01. Physics (Polytechnical Foundation)/Physics_Vol_II/01_repository/Physics_DTU/02_fall_semester/3rd_experiment/output
Part 1 runs found: []
Part 2 runs found: []


## Part 1: when does the hand lose contact?

What the code does for every run:

1. The first 0.4 s of the recording are quiet. Their mean sets the zero of the accelerometer and of the force probe. The push direction is counted positive for both.
2. **A** is the maximum of the acceleration. **B** is the first time after A where the acceleration is zero.
3. The **plateau** is the middle part of the slide, where only kinetic friction acts. Its mean is $a_f$, with uncertainty $\sigma/\sqrt{N}$. From it, $\mu_k=-a_f/g$.
4. **C** is the first time the acceleration enters the band $a_f\pm2\sigma$.
5. $t_{F0}$ is the first time after A where the hand force enters its noise band around zero. The force probe measures the contact directly, so this is the moment the hand lets go.
6. $a(F0)$ is the acceleration at $t_{F0}$. Its uncertainty combines the sensor noise and the timing between the two sensors: $\delta a=\sqrt{(\sigma/\sqrt{n})^2+(\dot a\,\delta t)^2}$ with $\delta t=\sqrt{(1/f_a)^2+(1/f_F)^2}$.
7. Standard scores compare $a(F0)$ with the value at A (the peak), at B (zero) and at C (the plateau). The hypothesis that describes the data gives $z\le2$. A clear rejection needs $z>3$.

Each run gets a plot. Check that A, B and C sit where you would put them by eye.

In [4]:
# ============================== PART 1: analyse every run ==============================
def load_part1_run(run_dir):
    acc = force = None
    for p in csv_files(run_dir):
        t, ch, names = read_iolab_csv(p)
        n = p.stem.lower()
        if "force" in n or (len(ch) == 1 and "acc" not in n):
            force = (t, ch[0], p.name)
        elif "acc" in n or looks_like_accelerometer(ch):
            acc = (t, ch, p.name)
    if acc is None or force is None:
        raise ValueError(f"need one accelerometer CSV and one force CSV, found {[p.name for p in csv_files(run_dir)]}")
    return acc, force


def pick_axis(t, chans):
    if ACC_AXIS in ("x", "y", "z"):
        return "xyz".index(ACC_AXIS)
    base = t < t[0] + BASELINE_S
    size = []
    for c in chans[:3]:
        c0 = np.mean(c[base])
        size.append(-1.0 if abs(c0) > 6 else float(np.max(np.abs(c - c0))))
    return int(np.argmax(size))


def analyse_part1(run_dir):
    (ta, chans, a_file), (tf, F_raw, f_file) = load_part1_run(run_dir)
    k = pick_axis(ta, chans)
    fa, fF = rate(ta), rate(tf)
    ba, bF = ta < ta[0] + BASELINE_S, tf < tf[0] + BASELINE_S
    a = chans[k] - np.mean(chans[k][ba])          # acceleration relative to rest (removes the sensor offset)
    F = F_raw - np.mean(F_raw[bF])                # software rezero
    notes = []
    if k != 1:
        notes.append(f"largest signal on the {'xyz'[k]} axis, not y")
    if np.std(a[ba], ddof=1) > 0.5 or np.std(F[bF], ddof=1) > 0.1:
        notes.append("start of recording is not quiet: wait about 1 s after Record before pushing")
    if a[np.argmax(np.abs(a))] < 0:               # count the push direction as positive
        a = -a
    if F[np.argmax(np.abs(F))] < 0:
        F = -F
    na = max(1, int(round(SMOOTH_MS / 1000 * fa)))
    nF = max(1, int(round(SMOOTH_MS / 1000 * fF)))
    a_s, F_s = moving_average(a, na), moving_average(F, nF)
    sa0 = np.std(a[ba], ddof=1)
    sF, sF_s = np.std(F[bF], ddof=1), np.std(F_s[bF], ddof=1)

    iA = int(np.argmax(a_s))                                    # A: peak
    tA, aA = ta[iA], a_s[iA]
    tB, iB = crossing_time(ta, a_s, 0.0, iA + 1, "down")        # B: first zero after the peak
    if tB is None:
        raise ValueError("the acceleration never goes below zero after the peak")
    thr = max(0.3, 5 * np.std(a_s[ba], ddof=1))                 # sliding phase: clearly below zero
    below = a_s < -thr
    if not below[iB:].any():
        raise ValueError("no friction plateau found after B")
    iS = iB + int(np.argmax(below[iB:]))
    need = max(2, int(0.03 * fa))
    iE = len(a_s) - 1
    for j in range(iS, len(a_s) - need):
        if np.all(a_s[j:j + need] > -thr):                      # it has stopped
            iE = j
            break
    tS, tE = ta[iS], ta[iE]
    p0, p1 = tS + 0.3 * (tE - tS), tE - 0.1 * (tE - tS)         # middle of the slide = plateau window
    plat = (ta >= p0) & (ta <= p1)
    if plat.sum() < 5:                                          # very short slide: use all of it
        p0, p1 = tS, tE
        plat = (ta >= p0) & (ta <= p1)
        notes.append("short slide: plateau taken over the whole slide")
    if plat.sum() < 3:
        raise ValueError("the slide is too short to measure the plateau; push a little harder")
    af, saf, Nf = np.mean(a[plat]), np.std(a[plat], ddof=1), int(plat.sum())
    daf = saf / math.sqrt(Nf)
    saf_s = np.std(a_s[plat], ddof=1)
    tC, _ = crossing_time(ta, a_s, af + BAND_K * saf_s, iB, "down")   # C: enters the plateau band
    if tC is None:
        tC = tS
    iAf = int(np.searchsorted(tf, tA))
    tF0, _ = crossing_time(tf, F_s, BAND_K * sF_s, iAf, "down")        # hand force reaches zero
    if tF0 is None:
        raise ValueError("the force never returns to zero after the push")

    dt_sync = math.sqrt((1 / fa) ** 2 + (1 / fF) ** 2)          # timing uncertainty between the two sensors
    w = max(SMOOTH_MS / 2000, 0.6 / fa)
    sel = np.abs(ta - tF0) <= w
    if not sel.any():
        sel[np.argmin(np.abs(ta - tF0))] = True
    aF0, nw = float(np.mean(a[sel])), int(sel.sum())
    slope_a = float(np.interp(tF0, ta, np.gradient(a_s, ta)))
    daF0 = math.sqrt((saf / math.sqrt(nw)) ** 2 + (slope_a * dt_sync) ** 2)

    dFdt = np.gradient(F_s, tf)

    def F_at(tx):
        return float(np.interp(tx, tf, F)), math.sqrt(sF ** 2 + (np.interp(tx, tf, dFdt) * dt_sync) ** 2)

    (FA, dFA), (FB, dFB), (FC, dFC) = F_at(tA), F_at(tB), F_at(tC)

    newton = None                                                # extra: a = F/m - mu_k g while sliding
    rise = np.nonzero(a_s[:iA + 1] > thr)[0]
    t_slide = ta[rise[0]] + 0.01 if rise.size else tA
    win = (ta >= t_slide) & (ta <= tF0)
    if win.sum() >= 6:
        Fw = np.interp(ta[win], tf, F)
        (sl, ic), cov = np.polyfit(Fw, a[win], 1, cov=True)
        dsl, dic = np.sqrt(np.diag(cov))
        newton = dict(m_fit=1 / sl, dm_fit=dsl / sl ** 2, intercept=ic, dintercept=dic, n=int(win.sum()),
                      Fw=Fw, aw=a[win])

    return dict(run=run_dir.name, files=(a_file, f_file), axis="xyz"[k], f_acc=fa, f_force=fF,
                tA=tA, tB=tB, tC=tC, tF0=tF0, tS=tS, tE=tE, p0=p0, p1=p1,
                aA=aA, af=af, saf=saf, saf_s=saf_s, daf=daf, Nf=Nf, mu=-af / g, dmu=daf / g,
                aF0=aF0, daF0=daF0, nw=nw, sa0=sa0, n0=int(ba.sum()),
                zA=zscore(aF0, daF0, aA, saf), zB=zscore(aF0, daF0, 0.0, sa0 / math.sqrt(ba.sum())),
                zC=zscore(aF0, daF0, af, daf),
                FA=FA, dFA=dFA, FB=FB, dFB=dFB, FC=FC, dFC=dFC, sF=sF, dt_sync=dt_sync,
                newton=newton, notes=notes, ta=ta, a=a, a_s=a_s, tf=tf, F=F, F_s=F_s)


def plot_part1(r, path=None, title=True):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.0, 4.6), sharex=True,
                                   gridspec_kw=dict(height_ratios=[1.25, 1], hspace=0.1))
    ax1.plot(r["ta"], r["a"], color=BLUE, lw=0.9)
    ax1.axvspan(r["p0"], r["p1"], color=AQUA, alpha=0.12, lw=0)
    ax1.axhline(r["af"], color=INK, lw=0.7, ls=":")
    ax2.plot(r["tf"], r["F"], color=ORANGE, lw=0.9)
    ax2.axhline(0, color=MUTED, lw=0.6)
    for tx, lab, ha in ((r["tA"], "A ", "right"), (r["tB"], "B ", "right"), (r["tC"], " C", "left")):
        for ax in (ax1, ax2):
            ax.axvline(tx, color=MUTED, lw=0.7, ls=(0, (4, 3)))
        ax1.text(tx, ax1.get_ylim()[1], lab, va="top", ha=ha, color=INK, fontsize=9)
    for ax in (ax1, ax2):
        ax.axvline(r["tF0"], color=ORANGE, lw=0.8, alpha=0.8)
    ax2.text(r["tF0"], ax2.get_ylim()[1], "  F reaches 0", va="top", ha="left", color=ORANGE, fontsize=8)
    ax1.set_ylabel(r"$a_y$ (m/s$^2$), push positive")
    ax2.set_ylabel(r"$F$ (N), push positive")
    ax2.set_xlabel("time $t$ (s)")
    ax1.set_xlim(r["tA"] - 0.25, r["tE"] + 0.15)
    if title:
        ax1.set_title(f"Part 1, {r['run']}: accelerometer {r['f_acc']:.0f} Hz, force {r['f_force']:.0f} Hz",
                      fontsize=9, loc="left")
    fig.align_ylabels((ax1, ax2))
    if path:
        fig.savefig(str(path) + ".png", dpi=300, bbox_inches="tight")
        fig.savefig(str(path) + ".pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)


res1 = []
for rd in run_folders(PART1_DIR):
    try:
        r = analyse_part1(rd)
        res1.append(r)
        print(f"{rd.name}: files {r['files']}, axis {r['axis']}"
              + ("" if not r["notes"] else "  NOTE: " + "; ".join(r["notes"])))
        plot_part1(r, OUT_DIR / f"part1_{rd.name}")
    except Exception as exc:
        print(f"{rd.name}: skipped ({exc})")

if res1:
    tab1 = pd.DataFrame([{
        "run": r["run"],
        "f_acc (Hz)": round(r["f_acc"]), "f_F (Hz)": round(r["f_force"]),
        "t_A (s)": round(r["tA"], 4), "t_B (s)": round(r["tB"], 4), "t_C (s)": round(r["tC"], 4),
        "t_F0 (s)": round(r["tF0"], 4),
        "t_F0 - t_B (ms)": round(1000 * (r["tF0"] - r["tB"]), 1),
        "t_F0 - t_C (ms)": round(1000 * (r["tF0"] - r["tC"]), 1),
        "a_A (m/s2)": round(r["aA"], 2),
        "a_f plateau (m/s2)": fmt(r["af"], r["daf"]),
        "a at F=0 (m/s2)": fmt(r["aF0"], r["daF0"]),
        "z vs A": round(r["zA"], 1), "z vs B": round(r["zB"], 1), "z vs C": round(r["zC"], 1),
        "F(t_B) (N)": fmt(r["FB"], r["dFB"]), "F(t_C) (N)": fmt(r["FC"], r["dFC"]),
        "mu_k": fmt(r["mu"], r["dmu"]),
    } for r in res1])
    tab1.to_csv(OUT_DIR / "part1_table.csv", index=False)
    show(tab1)
else:
    print("No Part 1 runs analysed. Put the CSV files in", PART1_DIR / "run1", "and so on, or use MANUAL_PART1.")

No Part 1 runs analysed. Put the CSV files in data/part1/run1 and so on, or use MANUAL_PART1.


### Part 1 summary over all runs

A mean over runs uses the standard deviation of the mean (SDOM) as its uncertainty, but never less than the typical single run uncertainty divided by $\sqrt{n}$. The Newton check at the end is an extra: while the hand pushes and the iOLab slides, $a=F/m-\mu_k g$, so a straight line fit of $a$ against $F$ has slope $1/m$ and intercept $-\mu_k g$.

In [5]:
# ============================== PART 1: summary over all runs ==============================
if res1:
    n1 = len(res1)
    dC = [r["aF0"] - r["af"] for r in res1]
    dC_unc = [math.sqrt(r["daF0"] ** 2 + r["daf"] ** 2) for r in res1]
    mC, uC, _ = mean_with_floor(dC, dC_unc)
    mF0, uF0, _ = mean_with_floor([r["aF0"] for r in res1], [r["daF0"] for r in res1])
    mAf, uAf, _ = mean_with_floor([r["af"] for r in res1], [r["daf"] for r in res1])
    mA, uA, _ = mean_with_floor([r["aA"] for r in res1], [r["saf"] for r in res1])
    mB0 = 0.0
    mmu, umu, _ = mean_with_floor([r["mu"] for r in res1], [r["dmu"] for r in res1])
    dtC = [1000 * (r["tF0"] - r["tC"]) for r in res1]
    dtB = [1000 * (r["tF0"] - r["tB"]) for r in res1]
    dtA = [1000 * (r["tF0"] - r["tA"]) for r in res1]
    res_ms = [1000 * r["dt_sync"] for r in res1]
    mtC, utC, _ = mean_with_floor(dtC, res_ms)
    mtB, utB, _ = mean_with_floor(dtB, res_ms)
    mtA, utA, _ = mean_with_floor(dtA, res_ms)
    mFB, uFB, _ = mean_with_floor([r["FB"] for r in res1], [r["dFB"] for r in res1])
    mFC, uFC, _ = mean_with_floor([r["FC"] for r in res1], [r["dFC"] for r in res1])

    print(f"PART 1 SUMMARY ({n1} runs)")
    print(f"  plateau (friction only)          a_f   = {fmt(mAf, uAf, 'm/s^2')}")
    print(f"  kinetic friction coefficient     mu_k  = {fmt(mmu, umu)}")
    print(f"  acceleration when F reaches 0    a(F0) = {fmt(mF0, uF0, 'm/s^2')}")
    print()
    print("  Where on the acceleration curve does the hand force reach zero?")
    zA_all = zscore(mF0, uF0, mA, uA)
    zB_all = zscore(mF0, uF0, 0.0, 0.0)
    zC_all = abs(mC) / uC if uC > 0 else float("inf")
    print(f"    compared with A (peak, {fmt(mA, uA, 'm/s^2')}):   z = {zA_all:6.1f}  -> {verdict(zA_all)}")
    print(f"    compared with B (a = 0):                  z = {zB_all:6.1f}  -> {verdict(zB_all)}")
    print(f"    compared with C (plateau a_f):            z = {zC_all:6.1f}  -> {verdict(zC_all)}"
          f"   [a(F0) - a_f = {fmt(mC, uC, 'm/s^2')}]")
    print()
    print("  Timing (F reaches 0) minus (point on the acceleration curve):")
    print(f"    t_F0 - t_A = {fmt(mtA, utA, 'ms')}")
    print(f"    t_F0 - t_B = {fmt(mtB, utB, 'ms')}   (z vs 0: {abs(mtB) / utB:.1f})")
    print(f"    t_F0 - t_C = {fmt(mtC, utC, 'ms')}   (z vs 0: {abs(mtC) / utC:.1f})")
    print(f"    one sample interval: {1000 / min(r['f_acc'] for r in res1):.2f} ms (accelerometer),"
          f" {1000 / min(r['f_force'] for r in res1):.2f} ms (force)")
    print()
    print("  Hand force at the marked points:")
    print(f"    F(t_B) = {fmt(mFB, uFB, 'N')}    theory m*mu_k*g = {fmt(m * mmu * g, g * math.sqrt((dm * mmu) ** 2 + (m * umu) ** 2), 'N')}")
    print(f"    F(t_C) = {fmt(mFC, uFC, 'N')}    theory 0")

    nt = [r["newton"] for r in res1 if r["newton"] is not None]
    if nt:
        mm, um, _ = mean_with_floor([x["m_fit"] for x in nt], [x["dm_fit"] for x in nt])
        mi, ui, _ = mean_with_floor([x["intercept"] for x in nt], [x["dintercept"] for x in nt])
        print()
        print("  Extra, Newton's 2nd law while the hand pushes (fit a = F/m + c):")
        print(f"    mass from the fit   m = {fmt(mm, um, 'kg')}   scale: {fmt(m, dm, 'kg')}   z = {zscore(mm, um, m, dm):.1f}")
        print(f"    intercept           c = {fmt(mi, ui, 'm/s^2')}   plateau a_f: {fmt(mAf, uAf, 'm/s^2')}   z = {zscore(mi, ui, mAf, uAf):.1f}")

    # Report figure: one clean run
    rr = next((r for r in res1 if r["run"] == REPORT_RUN_PART1), res1[0])
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6.3, 4.2), sharex=True,
                                   gridspec_kw=dict(height_ratios=[1.25, 1], hspace=0.1))
    ax1.plot(rr["ta"], rr["a"], color=BLUE, lw=1.0)
    ax2.plot(rr["tf"], rr["F"], color=ORANGE, lw=1.0)
    ax2.axhline(0, color=MUTED, lw=0.6)
    ax1.axhline(rr["af"], color=INK, lw=0.7, ls=":")
    for tx, lab, dx in ((rr["tA"], "A", -9), (rr["tB"], "B", -9), (rr["tC"], "C", 3)):
        for ax in (ax1, ax2):
            ax.axvline(tx, color=MUTED, lw=0.7, ls=(0, (4, 3)))
        ax1.annotate(lab, (tx, ax1.get_ylim()[1]), xytext=(dx, -12), textcoords="offset points", color=INK)
    ax1.set_ylabel(r"$a_y$ (m/s$^2$)")
    ax2.set_ylabel(r"$F_{hand}$ (N)")
    ax2.set_xlabel("time $t$ (s)")
    ax1.set_xlim(rr["tA"] - 0.2, rr["tE"] + 0.1)
    fig.align_ylabels((ax1, ax2))
    fig.savefig(OUT_DIR / "part1_report_figure.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUT_DIR / "part1_report_figure.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)

    if nt:
        fig, ax = plt.subplots(figsize=(4.2, 3.0))
        for x in nt:
            ax.plot(x["Fw"], x["aw"], ".", ms=3, color=BLUE, alpha=0.6)
        xx = np.linspace(0, max(np.max(x["Fw"]) for x in nt), 50)
        ax.plot(xx, xx / mm + mi, color=INK, lw=1)
        ax.set_xlabel(r"$F_{hand}$ (N)")
        ax.set_ylabel(r"$a_y$ (m/s$^2$)")
        ax.set_title("a = F/m + c while pushing (all runs)", fontsize=9, loc="left")
        fig.savefig(OUT_DIR / "part1_newton_fit.png", dpi=300, bbox_inches="tight")
        plt.show()
        plt.close(fig)

## Part 2: $a_1$ (up) and $a_2$ (down) on the ramp

What the code does for every run:

1. Up the ramp is counted positive, so $a_1$ and $a_2$ both come out negative (both point down the ramp).
2. The turning point is where the velocity crosses zero.
3. The up window starts when the push is over (the acceleration has settled, the Part 1 lesson) and ends 0.15 s before the turning point. The down window starts 0.15 s after the turning point and ends 0.10 s before you catch the iOLab.
4. $a_1$ and $a_2$ are the slopes of straight lines fitted to $v(t)$ in the two windows. The table also gives the iOLab app method: the mean of the acceleration graph with uncertainty $\sigma/\sqrt{N}$, $N=f\,\Delta t$.

Short windows (under about 0.4 s) give noisy slopes. If the plots show short windows, push harder or use a smaller angle.

In [6]:
# ============================== PART 2: analyse every run ==============================
import itertools

KIND_KEY = {"position": "r", "velocity": "v", "acceleration": "a"}


def classify_single(t, x):
    """Guess if one wheel column is position, velocity or acceleration.
    The right guess gives a steady negative acceleration in the middle of the motion."""
    if WHEEL_SINGLE_KIND in KIND_KEY:
        return WHEEL_SINGLE_KIND
    best, best_score = "velocity", np.inf
    for kind in KIND_KEY:
        if kind == "position":
            v = np.gradient(moving_average(x, 5), t)
            a = np.gradient(moving_average(v, 5), t)
        elif kind == "velocity":
            v, a = x, np.gradient(moving_average(x, 5), t)
        else:
            a = x
            v = np.concatenate([[0.0], np.cumsum(0.5 * (x[1:] + x[:-1]) * np.diff(t))])
        idx = np.nonzero(np.abs(v) > 0.05)[0]
        if idx.size < 10:
            continue
        s = 1.0 if v[idx[0]] > 0 else -1.0
        t0 = t[idx[0]] + 0.25 * (t[idx[-1]] - t[idx[0]])
        t1 = t[idx[0]] + 0.75 * (t[idx[-1]] - t[idx[0]])
        am = s * moving_average(a, 5)[(t >= t0) & (t <= t1)]
        if am.size < 5 or not (-5 < am.mean() < -0.05):
            continue
        score = am.std() / abs(am.mean())
        if score < best_score:
            best, best_score = kind, score
    return best


def identify_three(t, chans):
    """Order of (position, velocity, acceleration) among three columns, found from derivatives."""
    best, best_score = (0, 1, 2), -np.inf
    for perm in itertools.permutations(range(3)):
        r, v, a = (chans[i] for i in perm)
        c1 = np.corrcoef(np.gradient(moving_average(r, 3), t), v)[0, 1]
        c2 = np.corrcoef(np.gradient(moving_average(v, 3), t), a)[0, 1]
        score = np.nan_to_num(c1) + np.nan_to_num(c2)
        if score > best_score:
            best, best_score = perm, score
    return best


def load_part2_run(run_dir):
    wheel, acc, notes = {}, None, []
    for p in csv_files(run_dir):
        t, ch, names = read_iolab_csv(p)
        n = p.stem.lower()
        if looks_like_accelerometer(ch) or ("acc" in n and "wheel" not in n and len(ch) >= 3):
            acc = (t, ch)
            notes.append(f"{p.name}: accelerometer")
            continue
        if len(ch) >= 3:
            perm = identify_three(t, ch[:3])
            for kind, i in zip(KIND_KEY, perm):
                wheel[KIND_KEY[kind]] = (t, ch[i])
            notes.append(f"{p.name}: wheel columns {[names[i] for i in perm]} = position, velocity, acceleration")
        else:
            for j, c in enumerate(ch):
                kind = ("position" if re.search(r"pos|(^|_)ry", n) else "velocity" if re.search(r"vel|(^|_)vy", n)
                        else "acceleration" if re.search(r"acc|(^|_)ay", n) else classify_single(t, c))
                wheel[KIND_KEY[kind]] = (t, c)
                notes.append(f"{p.name}: column {names[j]} read as {kind}")
    if not wheel:
        raise ValueError(f"no wheel data found in {[p.name for p in csv_files(run_dir)]}")
    if "v" in wheel:
        t, v = wheel["v"]
    elif "r" in wheel:
        t, r = wheel["r"]
        v = np.gradient(moving_average(r, 5), t)
    else:
        t, a = wheel["a"]
        v = np.concatenate([[0.0], np.cumsum(0.5 * (a[1:] + a[:-1]) * np.diff(t))])
    if "a" in wheel:
        ta_, atr = wheel["a"]
    else:
        ta_, atr = t, np.gradient(moving_average(v, 5), t)
    return t, v, ta_, atr, acc, notes


def analyse_part2(run_dir):
    t, v, ta_, atr, acc, notes = load_part2_run(run_dir)
    f = rate(t)
    moving = np.nonzero(np.abs(v) > 0.1)[0]
    if moving.size == 0:
        raise ValueError("no motion found in the wheel data")
    s = 1.0 if v[moving[0]] > 0 else -1.0                  # up the ramp counted positive
    v, atr = s * v, s * atr
    vs = moving_average(v, 3)
    iv = int(np.argmax(vs))
    t_vmax = t[iv]
    t_turn, i_turn = crossing_time(t, vs, 0.0, iv + 1, "down")
    if t_turn is None:
        raise ValueError("velocity never crosses zero: the iOLab did not roll back down")
    i_min = i_turn + int(np.argmin(moving_average(v, 5)[i_turn:]))
    t_min = t[i_min]
    # Start of the up window: the hand is gone only when the acceleration has settled (same idea as point C in Part 1)
    a_sm = moving_average(np.interp(t, ta_, atr), 7)
    late = (t >= t_vmax + 0.5 * (t_turn - t_vmax)) & (t <= t_turn - MARGIN_TURN_S)
    t_start = t_vmax + MARGIN_PUSH_S
    if late.sum() >= 5:
        a1_est = np.median(a_sm[late])
        tol = max(0.05, 0.1 * abs(a1_est))
        settled = np.nonzero((t >= t_vmax) & (a_sm <= a1_est + tol))[0]
        if settled.size:
            t_start = max(t_start, t[settled[0]] + 0.03)
    up = (t_start, t_turn - MARGIN_TURN_S)
    dn = (t_turn + MARGIN_TURN_S, t_min - MARGIN_END_S)
    if run_dir.name in MANUAL_WINDOWS:
        up = MANUAL_WINDOWS[run_dir.name].get("up", up)
        dn = MANUAL_WINDOWS[run_dir.name].get("down", dn)
    for name, wdw in (("up", up), ("down", dn)):
        if wdw[1] - wdw[0] < 0.2:
            notes.append(f"{name} window only {1000 * (wdw[1] - wdw[0]):.0f} ms long: push harder or use a longer ramp")

    def slope_fit(wdw):
        sel = (t >= wdw[0]) & (t <= wdw[1])
        if sel.sum() < 5:
            return np.nan, np.nan, int(sel.sum())
        p, cov = np.polyfit(t[sel], v[sel], 1, cov=True)
        return p[0], math.sqrt(cov[0, 0]), int(sel.sum())

    def app_mean(wdw):
        sel = (ta_ >= wdw[0]) & (ta_ <= wdw[1])
        if sel.sum() < 3:
            return np.nan, np.nan, np.nan, 0
        mu, sd = atr[sel].mean(), atr[sel].std(ddof=1)
        N = rate(ta_) * (wdw[1] - wdw[0])
        return mu, sd, sd / math.sqrt(N), N

    a1, da1, n1 = slope_fit(up)
    a2, da2, n2 = slope_fit(dn)
    m1, s1, d1, N1 = app_mean(up)
    m2, s2, d2, N2 = app_mean(dn)
    out = dict(run=run_dir.name, f=f, t=t, v=v, ta=ta_, atr=atr, up=up, dn=dn, t_turn=t_turn,
               a1=a1, da1=da1, a2=a2, da2=da2, n1=n1, n2=n2,
               app1=(m1, s1, d1, N1), app2=(m2, s2, d2, N2), notes=notes, acc=None)

    if acc is not None:                                     # extra: 3D accelerometer on the ramp
        tq, ch = acc
        rest = tq < tq[0] + BASELINE_S
        k = int(np.argmax([abs(np.mean(c[rest])) if i != 2 else -1 for i, c in enumerate(ch[:3])]))
        ry, rz = ch[k], ch[2]
        sgn = 1.0 if np.mean(ry[rest]) > 0 else -1.0        # at rest the up-slope reading is +g sin(theta)
        ry = sgn * ry
        Ry0, dRy0 = np.mean(ry[rest]), np.std(ry[rest], ddof=1) / math.sqrt(rest.sum())
        Rz0, dRz0 = abs(np.mean(rz[rest])), np.std(rz[rest], ddof=1) / math.sqrt(rest.sum())
        th = math.atan2(Ry0, Rz0)
        dth = math.sqrt((Rz0 * dRy0) ** 2 + (Ry0 * dRz0) ** 2) / (Ry0 ** 2 + Rz0 ** 2)

        def win_mean(wdw):
            sel = (tq >= wdw[0]) & (tq <= wdw[1])
            return ry[sel].mean(), ry[sel].std(ddof=1) / math.sqrt(sel.sum())

        out["acc"] = dict(theta=th, dtheta=dth, R_up=win_mean(up), R_dn=win_mean(dn), axis="xyz"[k])
    return out


def plot_part2(r, path=None, title=True):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.0, 4.4), sharex=True,
                                   gridspec_kw=dict(hspace=0.1))
    for ax in (ax1, ax2):
        ax.axvspan(*r["up"], color=ORANGE, alpha=0.13, lw=0)
        ax.axvspan(*r["dn"], color=AQUA, alpha=0.13, lw=0)
        ax.axvline(r["t_turn"], color=MUTED, lw=0.7, ls=(0, (4, 3)))
    ax1.plot(r["t"], r["v"], color=BLUE, lw=1.0)
    ax1.axhline(0, color=MUTED, lw=0.6)
    for wdw, a_, c in ((r["up"], r["a1"], ORANGE), (r["dn"], r["a2"], AQUA)):
        sel = (r["t"] >= wdw[0]) & (r["t"] <= wdw[1])
        if sel.sum() >= 5:
            p = np.polyfit(r["t"][sel], r["v"][sel], 1)
            ax1.plot(r["t"][sel], np.polyval(p, r["t"][sel]), color=INK, lw=1.2)
    ax2.plot(r["ta"], r["atr"], color=BLUE, lw=0.9)
    ax2.hlines(r["a1"], *r["up"], color=INK, lw=1.2)
    ax2.hlines(r["a2"], *r["dn"], color=INK, lw=1.2)
    ax1.set_ylabel(r"$v$ (m/s), up positive")
    ax2.set_ylabel(r"$a$ (m/s$^2$)")
    ax2.set_xlabel("time $t$ (s)")
    lo = min(r["up"][0], r["dn"][0]) - 0.4
    hi = max(r["up"][1], r["dn"][1]) + 0.4
    ax1.set_xlim(lo, hi)
    sel = (r["ta"] >= lo) & (r["ta"] <= hi)
    mid = np.concatenate([r["atr"][(r["ta"] >= r["up"][0]) & (r["ta"] <= r["up"][1])],
                          r["atr"][(r["ta"] >= r["dn"][0]) & (r["ta"] <= r["dn"][1])]])
    if mid.size:
        ax2.set_ylim(min(mid.min(), r["a1"], r["a2"]) - 0.4, max(mid.max(), 0.0) + 0.5)
    if title:
        ax1.set_title(f"Part 2, {r['run']}: wheel {r['f']:.0f} Hz. Orange window = a1 (up), green = a2 (down)",
                      fontsize=9, loc="left")
    fig.align_ylabels((ax1, ax2))
    if path:
        fig.savefig(str(path) + ".png", dpi=300, bbox_inches="tight")
        fig.savefig(str(path) + ".pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)


res2 = []
for rd in run_folders(PART2_DIR):
    try:
        r = analyse_part2(rd)
        res2.append(r)
        print(f"{rd.name}: " + "; ".join(r["notes"]))
        plot_part2(r, OUT_DIR / f"part2_{rd.name}")
    except Exception as exc:
        print(f"{rd.name}: skipped ({exc})")

if res2:
    tab2 = pd.DataFrame([{
        "run": r["run"], "f (Hz)": round(r["f"]),
        "up window (s)": f"{r['up'][0]:.2f} to {r['up'][1]:.2f}",
        "a1 slope of v (m/s2)": fmt(r["a1"], r["da1"]),
        "a1 mean of a (m/s2)": fmt(r["app1"][0], r["app1"][2]),
        "down window (s)": f"{r['dn'][0]:.2f} to {r['dn'][1]:.2f}",
        "a2 slope of v (m/s2)": fmt(r["a2"], r["da2"]),
        "a2 mean of a (m/s2)": fmt(r["app2"][0], r["app2"][2]),
        "|a1|-|a2| (m/s2)": round(abs(r["a1"]) - abs(r["a2"]), 3),
    } for r in res2])
    tab2.to_csv(OUT_DIR / "part2_table.csv", index=False)
    show(tab2)
else:
    print("No Part 2 runs analysed. Put the CSV files in", PART2_DIR / "run1", "and so on, or use MANUAL_PART2.")

No Part 2 runs analysed. Put the CSV files in data/part2/run1 and so on, or use MANUAL_PART2.


### Part 2 summary over all runs

$a_1$ and $a_2$ are averaged over runs with the SDOM as uncertainty. Then:

* $|a_1|-|a_2|$ with $\delta=\sqrt{\delta a_1^2+\delta a_2^2}$ and its standard score test the hypothesis.
* $(|a_1|+|a_2|)/2$ cancels the friction and is compared with $g\sin\theta$ (needs `H`, `L` in the settings, or an accelerometer file where the iOLab rests still for the first 0.4 s).
* $(|a_1|-|a_2|)/2$ is the rolling resistance per mass, $F_r/m$.
* Extra: the 3D accelerometer along the ramp reads $a+g\sin\theta$, which leaves only the friction term: about $-F_r/m$ going up and $+F_r/m$ going down.

In [7]:
# ============================== PART 2: summary over all runs ==============================
def part2_summary(A1, A2, label, dA1=None, dA2=None):
    a1m, a1sd, _, n = mean_sd_sdom(A1)
    a2m, a2sd, _, _ = mean_sd_sdom(A2)
    _, a1u, _ = mean_with_floor(A1, dA1 if dA1 is not None else [0.0] * len(A1))
    _, a2u, _ = mean_with_floor(A2, dA2 if dA2 is not None else [0.0] * len(A2))
    D = abs(a1m) - abs(a2m)
    dD = math.sqrt(a1u ** 2 + a2u ** 2)
    z = abs(D) / dD if dD > 0 else float("inf")
    print(f"PART 2 SUMMARY ({label}, {n} runs; uncertainty = SDOM, at least the single run uncertainty / sqrt(n))")
    print(f"  a1 (up)   = {fmt(a1m, a1u, 'm/s^2')}    (SD of single runs {a1sd:.3f})")
    print(f"  a2 (down) = {fmt(a2m, a2u, 'm/s^2')}    (SD of single runs {a2sd:.3f})")
    print(f"  |a1| - |a2| = {fmt(D, dD, 'm/s^2')}  ->  z = {z:.1f}: a1 and a2 are {verdict(z)}")
    d = [abs(x) - abs(y) for x, y in zip(A1, A2)]
    dm_, dsd, du, _ = mean_sd_sdom(d)
    if np.isfinite(du) and du > 0:
        print(f"  paired, run by run: mean of |a1|-|a2| = {fmt(dm_, du, 'm/s^2')}  ->  z = {abs(dm_) / du:.1f}")
    return a1m, a1u, a2m, a2u, D, dD, z


def incline_from_ruler():
    if None in (H, dH, L, dL):
        return None
    s_ = H / L
    ds_ = s_ * math.sqrt((dH / H) ** 2 + (dL / L) ** 2)
    return math.asin(s_), ds_ / math.sqrt(1 - s_ ** 2)


if res2:
    A1 = [r["a1"] for r in res2]
    A2 = [r["a2"] for r in res2]
    a1m, a1u, a2m, a2u, D, dD, z2 = part2_summary(A1, A2, "slope of the velocity graph",
                                                   [r["da1"] for r in res2], [r["da2"] for r in res2])
    print()
    part2_summary([r["app1"][0] for r in res2], [r["app2"][0] for r in res2], "mean of the acceleration graph",
                  [r["app1"][2] for r in res2], [r["app2"][2] for r in res2])

    thetas = []
    th_r = incline_from_ruler()
    if th_r:
        thetas.append(("ruler", *th_r))
    acc_runs = [r["acc"] for r in res2 if r["acc"]]
    if acc_runs:
        tm, tsd, tu, na_ = mean_sd_sdom([x["theta"] for x in acc_runs])
        if na_ < 2 or not np.isfinite(tu):
            tu = acc_runs[0]["dtheta"]
        thetas.append(("accelerometer at rest", tm, tu))
    print()
    for name, th, dth in thetas:
        gs, dgs = g * math.sin(th), g * math.cos(th) * dth
        S, dS = (abs(a1m) + abs(a2m)) / 2, 0.5 * math.sqrt(a1u ** 2 + a2u ** 2)
        zS = zscore(S, dS, gs, dgs)
        print(f"  incline from {name}: theta = {fmt(math.degrees(th), math.degrees(dth), 'deg')}")
        print(f"    theory without friction: g sin(theta) = {fmt(gs, dgs, 'm/s^2')}")
        print(f"    (|a1| + |a2|)/2 = {fmt(S, dS, 'm/s^2')}  ->  z = {zS:.1f} ({verdict(zS)})")
        mu_r = D / (2 * g * math.cos(th))
        print(f"    rolling resistance per mass F_r/m = (|a1|-|a2|)/2 = {fmt(D / 2, dD / 2, 'm/s^2')}"
              f",  F_r = {fmt(m * D / 2, m * dD / 2, 'N')},  mu_r = {fmt(mu_r, dD / (2 * g * math.cos(th)))}")
    if acc_runs:
        up_ = [x["R_up"][0] for x in acc_runs]
        dn_ = [x["R_dn"][0] for x in acc_runs]
        um, _, uu, _ = mean_sd_sdom(up_)
        dmn, _, du2, _ = mean_sd_sdom(dn_)
        if not np.isfinite(uu):
            uu, du2 = acc_runs[0]["R_up"][1], acc_runs[0]["R_dn"][1]
        print()
        print("  Extra, 3D accelerometer along the ramp (reads a + g sin(theta), so only friction is left):")
        print(f"    going up   = {fmt(um, uu, 'm/s^2')}  (theory -F_r/m)")
        print(f"    going down = {fmt(dmn, du2, 'm/s^2')}  (theory +F_r/m)")
        print(f"    difference = {fmt(dmn - um, math.sqrt(uu ** 2 + du2 ** 2), 'm/s^2')}  compare |a1|-|a2| = {fmt(D, dD, 'm/s^2')}")

    rr = next((r for r in res2 if r["run"] == REPORT_RUN_PART2), res2[0])
    plot_part2(rr, OUT_DIR / "part2_report_figure", title=False)

    fig, ax = plt.subplots(figsize=(5.2, 3.0))
    x = np.arange(1, len(res2) + 1)
    ax.errorbar(x - 0.08, [abs(v) for v in A1], yerr=[r["da1"] for r in res2], fmt="o", ms=5, color=ORANGE,
                capsize=2, lw=1, label=r"$|a_1|$ up")
    ax.errorbar(x + 0.08, [abs(v) for v in A2], yerr=[r["da2"] for r in res2], fmt="s", ms=5, color=BLUE,
                capsize=2, lw=1, label=r"$|a_2|$ down")
    ax.axhspan(abs(a1m) - a1u, abs(a1m) + a1u, color=ORANGE, alpha=0.15, lw=0)
    ax.axhspan(abs(a2m) - a2u, abs(a2m) + a2u, color=BLUE, alpha=0.15, lw=0)
    ax.set_xticks(x)
    ax.set_xlabel("run")
    ax.set_ylabel(r"acceleration magnitude (m/s$^2$)")
    ax.legend(loc="best", fontsize=8, frameon=False)
    fig.savefig(OUT_DIR / "part2_runs_figure.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUT_DIR / "part2_runs_figure.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)

## Manual mode

Only used if you typed numbers into `MANUAL_PART1` or `MANUAL_PART2` in the settings. Every selection you make in the iOLab app gives $\mu$ (mean) and $\sigma$ (standard deviation). With the length $\Delta t$ of the selection and the sample rate $f$ the uncertainty of the mean is $\sigma/\sqrt{N}$ with $N=f\,\Delta t$.

In [8]:
# ============================== MANUAL MODE: numbers typed in from the iOLab app ==============================
if MANUAL_PART1:
    rows = []
    for d in MANUAL_PART1:
        f = d.get("f_acc", 100)
        mu0, sd0, w0 = d["a_at_F0"]
        muf, sdf, wf = d["plateau"]
        d0 = sd0 / math.sqrt(max(f * w0, 1))
        df_ = sdf / math.sqrt(max(f * wf, 1))
        rows.append(dict(run=d["run"], aF0=mu0, daF0=d0, af=muf, daf=df_, peak=d.get("peak", np.nan), sdf=sdf,
                         zA=zscore(mu0, d0, d.get("peak", np.nan), sdf), zB=zscore(mu0, d0, 0.0, 0.0),
                         zC=zscore(mu0, d0, muf, df_),
                         dtB=1000 * (d["tF0"] - d["tB"]) if "tB" in d else np.nan,
                         dtC=1000 * (d["tF0"] - d["tC"]) if "tC" in d else np.nan))
    man1 = pd.DataFrame(rows)
    show(man1.round(3))
    diff = man1["aF0"] - man1["af"]
    mC, sdC, uC, nC = mean_sd_sdom(diff)
    if nC < 2 or not np.isfinite(uC):
        uC = math.sqrt(man1["daF0"].iloc[0] ** 2 + man1["daf"].iloc[0] ** 2)
    print(f"MANUAL PART 1: a(F0) - a_f = {fmt(mC, uC, 'm/s^2')} -> z = {abs(mC) / uC:.1f} ({verdict(abs(mC) / uC)})")
    man1.to_csv(OUT_DIR / "part1_manual_table.csv", index=False)

if MANUAL_PART2:
    rows = []
    for d in MANUAL_PART2:
        f = d.get("f", 100)
        (m1, s1, w1), (m2, s2, w2) = d["up"], d["down"]
        N1, N2 = f * w1, f * w2
        rows.append(dict(run=d["run"], a1=m1, sd1=s1, N1=N1, da1=s1 / math.sqrt(N1),
                         a2=m2, sd2=s2, N2=N2, da2=s2 / math.sqrt(N2)))
    man2 = pd.DataFrame(rows)
    show(man2.round(4))
    man2.to_csv(OUT_DIR / "part2_manual_table.csv", index=False)
    if len(man2) >= 2:
        part2_summary(list(man2["a1"]), list(man2["a2"]), "manual, mean of the acceleration graph",
                      list(man2["da1"]), list(man2["da2"]))
    else:
        r0 = man2.iloc[0]
        D = abs(r0.a1) - abs(r0.a2)
        dD = math.sqrt(r0.da1 ** 2 + r0.da2 ** 2)
        print(f"ONE RUN ONLY: |a1| - |a2| = {fmt(D, dD, 'm/s^2')} -> z = {abs(D) / dD:.1f} ({verdict(abs(D) / dD)})")

if not (MANUAL_PART1 or MANUAL_PART2):
    print("Manual mode not used.")

Manual mode not used.


## What to take into the report

From the `output` folder:

1. `part1_report_figure.png`: acceleration and force against time for one run, with A, B and C marked. Add your own caption.
2. `part1_table.csv`: one row per run. Copy the columns you need into a table with units.
3. `part2_report_figure.png`: velocity and acceleration for one run with the two windows.
4. `part2_runs_figure.png`: $|a_1|$ and $|a_2|$ for every run with error bars.
5. `part2_table.csv`, and the printed summaries above (means, uncertainties, standard scores, $g\sin\theta$).

Round every number with its uncertainty, keep the units, and say in the text which run the figures show.